# Week11 Capstone Project - Cluster Centroid Refinement

## Strategy
For Week 11, the search is viewed through a **clustering lens**. Instead of only following the single best point, the method identifies a small cluster of the strongest recent weekly points and uses their **centroid** as a stable summary of the promising region.

For each function, the notebook:
1. loads the starter `.npy` data and the historical record from `Week11.csv`
2. takes the top 4 historical weekly points by output and computes their centroid
3. builds a small human-clean candidate pool around the centroid, best point, latest point, and second-best point
4. fits a GP surrogate and scores candidates with a local UCB-style rule
5. chooses the best candidate and reports which local cluster it targets

Scoring rule:

$$\text{score}(x)=\mu(x)+\beta(d)\sigma(x)-0.14\,\text{dist}(x,\text{centroid})-0.08\,\text{dist}(x,\text{best})-0.04\,\text{dist}(x,\text{latest})+0.02\,\text{novelty}(x)$$

with

$$\beta(d)=0.10+0.04d$$

This encourages refinement inside a promising cluster while still allowing mild uncertainty-driven exploration.


In [4]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


In [5]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week11.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
64,9,1,2,0.62800,0.63200,NaN,NaN,NaN,NaN,NaN,NaN,1.941930
65,9,2,2,0.49000,0.71000,NaN,NaN,NaN,NaN,NaN,NaN,0.542411
66,9,3,3,0.37200,0.56800,0.47200,NaN,NaN,NaN,NaN,NaN,-0.004842
67,9,4,4,0.39650,0.39350,0.35900,0.44050,NaN,NaN,NaN,NaN,0.217208
68,9,5,4,0.99999,0.00001,0.99999,0.99999,NaN,NaN,NaN,NaN,4440.106253
69,9,6,5,0.52500,0.17500,0.62500,0.87500,0.0250,NaN,NaN,NaN,-0.495826
70,9,7,6,0.01450,0.18150,0.28950,0.17250,0.3980,0.6660,NaN,NaN,2.099082
71,9,8,8,0.03480,0.43820,0.01180,0.32380,0.9905,0.0462,0.0978,0.7032,9.590284


In [6]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history):
    X0, y0 = load_initial(function_id)
    rows = history[history['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, Xw, yw, d, rows

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [7]:
cfg = {
    1: {'grid': 0.001, 'step': 0.002},
    2: {'grid': 0.005, 'step': 0.010},
    3: {'grid': 0.001, 'step': 0.002},
    4: {'grid': 0.0005, 'step': 0.0010},
    5: {'grid': 0.00001, 'step': 0.00001},
    6: {'grid': 0.001, 'step': 0.002},
    7: {'grid': 0.0005, 'step': 0.0005},
    8: {'grid': 0.0001, 'step': 0.0002},
}

def beta_by_dimension(d):
    return 0.10 + 0.04 * d

def top_cluster_centroid(Xw, yw, top_k=4):
    idx = np.argsort(yw)[-min(top_k, len(yw)):]
    return Xw[idx].mean(axis=0)

def build_gp(seed, d):
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.01, 1.5))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed,
        n_restarts_optimizer=1,
    )

def build_cluster_candidates(best_x, latest_x, second_x, centroid, d, step, grid):
    cands = []
    anchors = [
        best_x,
        latest_x,
        second_x,
        centroid,
        (best_x + centroid) / 2.0,
        (latest_x + centroid) / 2.0,
        (2 * best_x + centroid) / 3.0,
    ]
    for a in anchors:
        cands.append(round_grid(a, grid))
    for anchor in [best_x, centroid]:
        for i in range(d):
            for s in (-1.0, 1.0):
                x = anchor.copy()
                x[i] += s * step
                cands.append(round_grid(x, grid))
    for i in range(min(d, 4)):
        for j in range(i + 1, min(d, 4)):
            x = centroid.copy()
            x[i] += step
            x[j] -= step
            cands.append(round_grid(x, grid))
            x = centroid.copy()
            x[i] -= step
            x[j] += step
            cands.append(round_grid(x, grid))
    return np.unique(np.array(cands), axis=0)


In [8]:
def suggest_week11_point(function_id, history, seed=1200):
    X, y, Xw, yw, d, rows = load_combined(function_id, history)
    best_idx = int(np.argmax(yw))
    second_idx = int(np.argsort(yw)[-2])
    best_x = Xw[best_idx].copy()
    second_x = Xw[second_idx].copy()
    latest_x = Xw[-1].copy()
    centroid = top_cluster_centroid(Xw, yw, top_k=4)
    grid = cfg[function_id]['grid']
    step = cfg[function_id]['step']
    beta = beta_by_dimension(d)
    cands = build_cluster_candidates(best_x, latest_x, second_x, centroid, d, step, grid)
    dist_hist = np.sqrt(((cands[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist_hist.min(axis=1)
    novelty_floor = grid / 2 if grid < 0.001 else 0.0005
    keep = min_dist >= novelty_floor
    cands = cands[keep]
    min_dist = min_dist[keep]
    gp = build_gp(seed + function_id, d)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)
    mean, std = gp.predict(cands, return_std=True)
    centroid_dist = np.sqrt(((cands - centroid) ** 2).sum(axis=1)) / np.sqrt(d)
    best_dist = np.sqrt(((cands - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((cands - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)
    score = mean + beta * std - 0.14 * centroid_dist - 0.08 * best_dist - 0.04 * latest_dist + 0.02 * min_dist
    idx = int(np.argmax(score))
    return {
        'function': function_id,
        'd': d,
        'x': cands[idx],
        'formatted': format_point(cands[idx]),
        'cluster_centroid': format_point(centroid),
        'best_anchor': format_point(best_x),
        'latest_anchor': format_point(latest_x),
        'distance_to_centroid': float(centroid_dist[idx]),
    }


In [ ]:
results = []
for function_id in range(1, 9):
    r = suggest_week11_point(function_id, history)
    print(f"Function {function_id}: {r['formatted']}")
    print(f"  cluster centroid: {r['cluster_centroid']}")
    print(f"  best anchor:      {r['best_anchor']}")
    print(f"  latest anchor:    {r['latest_anchor']}")
    print(f"  dist to centroid: {r['distance_to_centroid']:.6f}")
    print()
    row = {'function': r['function'], 'd': r['d']}
    for i, v in enumerate(r['x'], start=1):
        row[f'x{i}'] = round(float(v), 6)
    results.append(row)
week11_df = pd.DataFrame(results)
week11_df.to_csv('Week11_suggestions.csv', index=False)
week11_df
